In [1]:
# Run garbage collection
import gc
gc.collect()

13

In [2]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import wfdb
import ast
import torch
import seaborn as sns
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, confusion_matrix
from tabulate import tabulate
import os
from collections import Counter
from sklearn.preprocessing import StandardScaler

In [3]:
import scipy.signal as signal
def filter_signal(signal_data, axis=0, fs=100, lowcut=0.5, highcut=45.0, notch_freq=50.0, Q=30.0):
        nyquist = 0.5 * fs
        low = lowcut / nyquist
        high = highcut / nyquist
        b, a = signal.butter(4, [low, high], btype='band')
        notch_b, notch_a = signal.iirnotch(notch_freq / nyquist, Q)
        filtered_signal = signal.filtfilt(b, a, signal_data, axis=axis)
        filtered_signal = signal.filtfilt(notch_b, notch_a, filtered_signal, axis=axis)
        return filtered_signal

In [4]:
def normalize_data_z_score(data: np.ndarray) -> np.ndarray:
    num_samples, sample_length, num_channels = data.shape
    data_reshaped = data.reshape(-1, num_channels)
    scaler = StandardScaler()
    data_normalized = scaler.fit_transform(data_reshaped)
    return data_normalized.reshape(num_samples, sample_length, num_channels)

In [5]:
path_load = './data_source_sub'


##train
X_train = np.load(os.path.join(path_load, 'Y_train.npy'))
X_train = normalize_data_z_score(X_train)
X_train = filter_signal(X_train, axis=1, fs=100, lowcut=0.5, highcut=45.0)

y_train = pd.read_csv(os.path.join(path_load, 'Z_train.csv'))
y_train = y_train.drop(columns=['ecg_id'])
y_train = y_train.to_numpy(dtype=np.float32)

z_train = pd.read_csv(os.path.join(path_load, 'T_train.csv'))
z_train = z_train.drop(columns=['ecg_id'])
z_train = z_train.to_numpy(dtype=np.float32)

##test
X_test = np.load(os.path.join(path_load, 'Y_test.npy'))
X_test = normalize_data_z_score(X_test)
X_test = filter_signal(X_test, axis=1, fs=100, lowcut=0.5, highcut=45.0)

y_test = pd.read_csv(os.path.join(path_load, 'Z_test.csv'))
y_test = y_test.drop(columns=['ecg_id'])
y_test = y_test.to_numpy(dtype=np.float32)

z_test = pd.read_csv(os.path.join(path_load, 'T_test.csv'))
z_test = z_test.drop(columns=['ecg_id'])
z_test = z_test.to_numpy(dtype=np.float32)

##validation
X_val = np.load(os.path.join(path_load, 'Y_valid.npy'))
X_val = normalize_data_z_score(X_val)
X_val = filter_signal(X_val, axis=1, fs=100, lowcut=0.5, highcut=45.0)

y_val = pd.read_csv(os.path.join(path_load, 'Z_valid.csv'))
y_val = y_val.drop(columns=['ecg_id'])
y_val = y_val.to_numpy(dtype=np.float32)

z_val = pd.read_csv(os.path.join(path_load, 'T_valid.csv'))
z_val = z_val.drop(columns=['ecg_id'])
z_val = z_val.to_numpy(dtype=np.float32)

table = [
    ["X_train", X_train.shape],
    ["y_train", y_train.shape],
    ["z_train", z_train.shape],
    ["X_test", X_test.shape],
    ["y_test", y_test.shape],
    ["z_test", z_test.shape],
    ["X_val", X_val.shape],
    ["y_val", y_val.shape],
    ["z_val", z_val.shape],
]

print(tabulate(table, headers=["Dataset", "Shape"], tablefmt="grid"))

+-----------+-------------------+
| Dataset   | Shape             |
+===========+===================+
| X_train   | (17441, 1000, 12) |
+-----------+-------------------+
| y_train   | (17441, 5)        |
+-----------+-------------------+
| z_train   | (17441, 23)       |
+-----------+-------------------+
| X_test    | (2203, 1000, 12)  |
+-----------+-------------------+
| y_test    | (2203, 5)         |
+-----------+-------------------+
| z_test    | (2203, 23)        |
+-----------+-------------------+
| X_val     | (2193, 1000, 12)  |
+-----------+-------------------+
| y_val     | (2193, 5)         |
+-----------+-------------------+
| z_val     | (2193, 23)        |
+-----------+-------------------+


In [6]:
def detect_r_peaks(ecg_, fs):
    """
    Simple R-peak detector using diff, squaring, moving window integration, and adaptive thresholding.
    Returns: indices of detected R-peaks.
    """
    def simple_diff(signal):
        diff = []
        for i in range(1, len(signal)):
            diff.append(signal[i] - signal[i-1])
        return np.array(diff)

    diff = simple_diff(ecg_)
    diff = np.append(diff, 0)  # Append zero to match original length

    # Step 2: Squaring
    squared = diff ** 2

    # Step 3: Moving window integration
    window_size = int(0.150 * fs)  # 150 ms window
    integrated = np.convolve(squared, np.ones(window_size) / window_size, mode='same')

    # Step 4: Simple adaptive thresholding and refractory period
    threshold = np.mean(integrated) * 1.5
    refractory_period = int(0.10 * fs)  # 250 ms

    peaks = []
    last_peak = -refractory_period

    for i in range(1, len(integrated) - 1):
        if integrated[i] > threshold and integrated[i] > integrated[i - 1] and integrated[i] > integrated[i + 1]:
            if i - last_peak > refractory_period:
                # Local search in raw signal for actual peak in a small window
                window = ecg_[i-10:i+10]
                if len(window) == 20:
                    true_peak = i - 10 + np.argmax(window)
                    peaks.append(true_peak)
                    last_peak = true_peak
    return np.array(peaks), integrated, squared, diff

def segment_PQRST(ecg, r_peaks, start_idx=40, end_idx=100):
    # The shape of ecg is (1, 1000, 12), so length is 1000 and number of channels is 12
    length = 1000
    PQRSTs = []
    expected_segment_length = start_idx + end_idx
    plt.figure()
    # Iterate through each lead (channel) first
    for lead_idx in range(ecg.shape[2]):  # Loop through each lead (channel)
        segments_for_this_lead = []
        #plt.figure()
        # Iterate through each R-peak and extract segments for this lead
        for i, r_peak in enumerate(r_peaks):
            start_beat = np.max([0, r_peak - start_idx])
            end_beat = np.min([r_peak + end_idx, length])
            if end_beat == length:
                continue
            # Check if we need padding at the start of the segment (if r_peak - start_idx is negative)
            if r_peak - start_idx < 0:
                start_padding = np.zeros(abs(r_peak - start_idx))  # Create padding at the start
                segment = ecg[0, start_beat:end_beat, lead_idx]  # Extract the segment
                segment = np.concatenate((start_padding, segment))  # Add start padding
            else:
                # Extract the segment without start padding if no negative index
                segment = ecg[0, start_beat:end_beat, lead_idx]
            
            # Pad the segment if it's shorter than the expected segment length at the end
            if expected_segment_length > len(segment):
                padding = np.zeros(expected_segment_length - len(segment))  # Create padding
                segment = np.concatenate((segment, padding))  # Pad the segment

            # Make sure the segment has the expected length
            assert len(segment) == expected_segment_length, f"Segment length mismatch: {len(segment)} != {expected_segment_length}"

            segments_for_this_lead.append(segment)
            
            # Plotting the segment for this lead (optional)
            plt.plot(segment)
        # plt.title(f"Lead {lead_idx+1}, R-peak {i+1}")
        # plt.xlabel('Time')
        # plt.ylabel('Amplitude')
        # plt.grid()
        
        # Add the segments for this lead (as a 2D array)
        PQRSTs.append(np.array(segments_for_this_lead))
    #plt.show()
    return PQRSTs

def compute_mean_std(PQRSTs):
    # Convert PQRSTs into a NumPy array (12 leads, 4 R-peaks, 250 samples per segment)
    PQRSTs = np.array(PQRSTs)  # Shape: (12, 4, 250)
    
    # Compute mean and std for each lead across R-peaks (axis 1) while preserving the time axis (axis 2)
    means = np.mean(PQRSTs, axis=1, keepdims=True)  # Mean across R-peaks (12, 1, 250)
    stds = np.std(PQRSTs, axis=1, keepdims=True)    # Std across R-peaks (12, 1, 250)
    
    return means, stds

In [7]:
def find_wave_point(signal, start, end, find_min=True):
    segment = signal[start:end]
    if find_min:
        idx = segment.argmin()
    else:
        idx = segment.argmax()
    return start + idx

def detect_qrst(signal, r_peaks, fs=100):
    peaks = []
    duration = {"Q": 0.08,
                "S": 0.08,
                "P": 0.3,
                "T": 0.4}
    for i, r in enumerate(r_peaks):
        q_search_start = max(r - int(duration["Q"] * fs), 0)
        q_search_end = r
        s_search_start = r
        s_search_end = min(r + int(duration["S"] * fs), len(signal))

        p_search_start = max(q_search_start - int(duration["P"] * fs), 0)
        p_search_end = q_search_start

        t_search_start = s_search_end
        t_search_end = min(s_search_end + int(duration["T"] * fs), len(signal))

        # Find Q, S, P, T waves
        q = find_wave_point(signal, q_search_start, q_search_end, find_min=True)
        s = find_wave_point(signal, s_search_start, s_search_end, find_min=True)
        p = find_wave_point(signal, p_search_start, p_search_end, find_min=False)
        t = find_wave_point(signal, t_search_start, t_search_end, find_min=False)
        
        peaks.append({
            'P': p,
            'Q': q,
            'R': r,
            'S': s,
            'T': t,
        })
    return peaks

def plot_intervals(signal, intervals, fs):
    plt.figure(figsize=(12, 6))
    t = np.arange(len(signal)) / fs
    plt.plot(t, signal, label='ECG Signal')

    for interval in intervals:
        r = interval['R'] / fs
        p = interval['P'] / fs
        q = interval['Q'] / fs
        s = interval['S'] / fs
        t_wave = interval['T'] / fs
        
        plt.plot([p, q], [signal[interval['P']], signal[interval['Q']]], 'ro-')
        plt.plot([q, r], [signal[interval['Q']], signal[interval['R']]], 'go-')
        plt.plot([r, s], [signal[interval['R']], signal[interval['S']]], 'bo-')
        plt.plot([s, t_wave], [signal[interval['S']], signal[interval['T']]], 'mo-')

    plt.title('ECG Signal with Intervals')
    plt.xlabel('Time (s)')
    plt.ylabel('Amplitude')
    plt.legend(['ECG Signal', 'P-Q', 'Q-R', 'R-S', 'S-T'])
    plt.grid()
    plt.show()

In [8]:
import numpy as np
from scipy.signal import savgol_filter
import matplotlib.pyplot as plt


def detect_pqst_slope(signal, r_peaks, fs):
    """
    Low-compute ECG delineation:
    Savitzky–Golay smoothing -> derivative -> adaptive thresholds.
    
    Parameters
    ----------
    signal : 1D np.array
        Raw ECG signal (float).
    r_peaks : list[int]
        Indices of detected R peaks.
    fs : int or float
        Sampling frequency in Hz.
        
    Returns
    -------
    results : dict
        Dictionary with lists for P, Q, S, T peaks (indices).
    """
    # --- Step 1: Smooth for clean derivative
    smoothed = savgol_filter(signal, window_length=11, polyorder=3)

    # --- Step 2: Derivative for slope info
    deriv = np.gradient(smoothed)

    waves = []
    P_waves, Q_waves, S_waves, T_waves = [], [], [], []
    duration = {"P": 0.3,
                 "Q": 0.08,
                 "S": 0.08,
                 "T": 0.4}
    for r in r_peaks:
        r_amp = smoothed[r]

        # --- Step 3: Find Q (max slope before R crossing below alpha*R_amp)
        q_search_start = max(0, r - int(duration["Q"] * fs))  # 50 ms before R
        q_search_end   = r
        if q_search_end > q_search_start:
            q_idx_rel = np.argmin(smoothed[q_search_start:q_search_end])
            q_idx = q_search_start + q_idx_rel
            Q_waves.append(q_idx)
        else:
            q_idx = r
            Q_waves.append(r)

        # --- Step 4: Find S (after R)
        s_search_start = r
        s_search_end = min(len(smoothed), r + int(duration["S"] * fs))
        
        if s_search_end > s_search_start:  # Check if the window is valid
            s_idx_rel = np.argmin(smoothed[s_search_start:s_search_end])
            s_idx = s_search_start + s_idx_rel
            S_waves.append(s_idx)
        else:
            s_idx = r
            S_waves.append(r)  # If window is invalid, use R as S

        # --- Step 5: Find P (low amp peak before Q, ~120 ms window)
        p_search_start = max(0, q_idx - int(duration["P"] * fs))
        p_search_end   = q_idx - int(0.04 * fs)  # avoid overlap
        if p_search_end > p_search_start:
            p_idx_rel = np.argmax(smoothed[p_search_start:p_search_end])
            p_idx = p_search_start + p_idx_rel
            P_waves.append(p_idx)
        else:
            p_idx = 1
            P_waves.append(1)

        # --- Step 6: Find T (after S, ~200–400 ms window)
        t_search_start = s_idx + int(0.08 * fs)
        t_search_end   = min(len(smoothed), s_idx + int(duration["T"] * fs))
        if t_search_end > t_search_start:
            t_idx_rel = np.argmax(smoothed[t_search_start:t_search_end])
            t_idx = t_search_start + t_idx_rel
            T_waves.append(t_idx)
        else:
            t_idx = len(signal)-1
            T_waves.append(len(signal)-1)
            
        waves.append({"P": p_idx,
                     "Q": q_idx,
                     "R": r,
                     "S": s_idx,
                     "T": t_idx})

    intervals = {
            "P": P_waves,
            "Q": Q_waves,
            "R": r_peaks,
            "S": S_waves,
            "T": T_waves,
        }
    return waves, intervals

def segment_PR_QRS_ST_QT(ecg, waves):
    """
    Segment PR, QRS, ST, and QT intervals per lead into fixed shapes.

    Parameters
    ----------
    ecg : np.ndarray
        Shape (1, length, 12), single ECG record.
    waves : list[dict]
        Output from detect_pqst_slope: each dict has keys 'P','Q','R','S','T'.

    Returns
    -------
    segments : dict
        Keys: 'PR', 'QRS', 'ST', 'QT'.
        Shapes: PR -> (12, B, 30), QRS -> (12, B, 50),
                ST -> (12, B, 100), QT -> (12, B, 200)
    """
    length = ecg.shape[1]
    n_beats = len(waves)
    n_leads = ecg.shape[2]

    # Preallocate zero arrays
    PR_arr  = np.zeros((n_leads, n_beats, 100))
    QRS_arr = np.zeros((n_leads, n_beats, 100))
    ST_arr  = np.zeros((n_leads, n_beats, 100))
    QT_arr  = np.zeros((n_leads, n_beats, 100))

    for lead_idx in range(n_leads):
        for beat_idx, beat in enumerate(waves):
            P, Q, R, S, T = beat["P"], beat["Q"], beat["R"], beat["S"], beat["T"]

            # --- PR interval
            PR_seg = ecg[0, P:Q, lead_idx]
            PR_seg = PR_seg[:100]  # truncate if longer
            PR_arr[lead_idx, beat_idx, :len(PR_seg)] = PR_seg

            # --- QRS interval
            QRS_seg = ecg[0, Q:S, lead_idx]
            QRS_seg = QRS_seg[:100]
            QRS_arr[lead_idx, beat_idx, :len(QRS_seg)] = QRS_seg

            # --- ST interval
            ST_seg = ecg[0, S:T, lead_idx]
            ST_seg = ST_seg[:100]
            ST_arr[lead_idx, beat_idx, :len(ST_seg)] = ST_seg

            # --- QT interval
            QT_seg = ecg[0, Q:T, lead_idx]
            QT_seg = QT_seg[:100]
            QT_arr[lead_idx, beat_idx, :len(QT_seg)] = QT_seg

    segments = {
        "PR": PR_arr,
        "QRS": QRS_arr,
        "ST": ST_arr,
        "QT": QT_arr,
    }
    np_segments = np.stack((PR_arr, QRS_arr, ST_arr, QT_arr), axis=1)
    # for k, v in segments.items():
    #     print(f"{k}: {v.shape}")
    return segments, np_segments


In [9]:
# ecg = X_test[:, :, :]
# 
# classes = 0
# indices = np.where(y_test[:, classes] == 1.0)[0]
# index = np.random.choice(indices)
# fs = 100
# ecg_choose = ecg[index, :, :]
# ecg_leadII = ecg_choose[:, 1]
# r_peaks, _, _, _ = detect_r_peaks(ecg_leadII, fs)
# ecg_choose = np.expand_dims(ecg_choose, axis=0)
# waves, intervals = detect_pqst_slope(ecg_leadII, r_peaks, fs)
# segments,np_segments = segment_PR_QRS_ST_QT(ecg_choose, waves)


In [10]:
ECG_TO_VCG = np.array([
        [ 0.217,  0.052, -0.038, -0.089, -0.098,  0.119, -0.202,  0.400,  0.125,  0.079, -0.356,  0.306],
        [-0.384, -0.457,  0.157,  0.162,  0.025, -0.236, -0.485, -0.286,  0.471,  0.117,  0.107, -0.096],
        [-0.013, -0.187,  0.020,  0.574, -0.118, -0.116, -0.121, -0.004,  0.064, -0.482, -0.203,  0.020]
    ])

def compute_vcg_mean_std(ecg_segments):
    """
    ecg_segments: (8, 4, B, L)  # leads, feature, beats, length
    Returns: (3, 4, 2, L)       # VCG xyz, feature, mean/std, length
    """
    vcg_features = []

    for feat_idx in range(ecg_segments.shape[1]):  # PR, QRS, ST, QT
        # Extract: (8, B, L)
        ecg_feat = ecg_segments[:, feat_idx, :, :]

        # Move leads to last axis → (B, L, 12)
        ecg_feat_reordered = np.moveaxis(ecg_feat, 0, -1)

        # Apply ECG→VCG: (B, L, 12) @ (12, 3) → (B, L, 12)
        vcg_feat = ecg_feat_reordered @ ECG_TO_VCG.T

        # Move VCG axis first → (3, B, L)
        vcg_feat = np.moveaxis(vcg_feat, -1, 0)

        # Compute mean and std over beats axis
        mean_feat = np.mean(vcg_feat, axis=1)  # (3, L)
        std_feat = np.std(vcg_feat, axis=1)    # (3, L)

        # Stack mean & std → (3, 2, L)
        vcg_features.append(np.stack([mean_feat, std_feat], axis=1))
    results = np.stack(vcg_features, axis=1)  # (3, 4, 2, L)
    results = np.reshape(results, (8, 100, 3))
    results = np.expand_dims(results, axis=0)
    return results


In [11]:
def compute_vcg_pr_qrs_st_qt(dataset, fs = 100):
    # Shape: (3, 12)  -> 3 VCG leads from 12 ECG leads
    all_vcg = []
    for i in range(dataset.shape[0]):
        ecg = dataset[i]
        ecg = np.expand_dims(ecg, axis = 0)
        ecg_leadII = ecg[0, :, 1]
        r_peaks, _, _, _ = detect_r_peaks(ecg_leadII, fs)
        waves, _ = detect_pqst_slope(ecg_leadII, r_peaks, fs)
        _,np_segments = segment_PR_QRS_ST_QT(ecg, waves)
        vcg_features = compute_vcg_mean_std(np_segments)
        all_vcg.append(vcg_features)
    all_vcg = np.concatenate(all_vcg, axis=0)
    return all_vcg

In [12]:
#data = compute_vcg_pr_qrs_st_qt(X_test)

import os
path = "./data_source_sub/"
data_test = compute_vcg_pr_qrs_st_qt(X_test,fs=100)
data_train = compute_vcg_pr_qrs_st_qt(X_train,fs=100)
data_val = compute_vcg_pr_qrs_st_qt(X_val,fs=100)

np.save(os.path.join(path, "M_val.npy"), data_val)
np.save(os.path.join(path, "M_test.npy"), data_test)
np.save(os.path.join(path, "M_train.npy"), data_train)
